In [1]:
from  pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
from pyspark.sql.functions import abs

In [ ]:
buildings_path = "Buildings.csv"
hostenergyPath = "HistoryEnergyConsumption.csv"
forecastPath = "EnergyConsumptForecast.csv"

#Ex1
def calculate_percentage(historical, forecast):
    return (abs(forecast – historical) / historical * 100)

buildings = spark.read.laod(buildings_path, format = "csv", header = true, inferSchema = True)
histEn = spark.read.laod(hostenergyPath, format = "csv", header = true, inferSchema = True)
forecastEn = spark.read.laod(forecastPath, format = "csv", header = true, inferSchema = True)

histEn = histEn.where("EnergykWH > 0")
joinedConsum = histEn.join(forecastEn, ["BuildingID", "Timestamp"], "inner")
spark.udf.register("calculate_abs", calculate_percentage)

joinedConsum  =joinedConsum.selectExpr("BuildingID", "Timestamp", "calculate_abs(EnergykWH, ForecastkWH) as AbsValue")
joinedConsum = joinedConsum.where("AbsValue > 20")

finalDF = joinedConsum.join(buildings, "BuildingID", "inner")
finalDF = finalDF.select("Type","EnergyEfficiency", "Timestamp").groupby("Type", "EnergyEfficiency").agg(count("Timestamp").alias("TimestampsAboveThreshold")) 
finalDF.writecsv(outputPath1, header = True)

#Ex2
def get_year(timestamp: str):
    return timestamp.split("/")[0] 

spark.udf.register("get_year", get_year)
joinedConsum = histEn.join(buildings, 'BuildingID', 'left')
joinedConsum = joinedConsum.selectExpr("BuildingID", "Timestamp", "EnergyKwh/SquareMeters as Ratio")
joinedConsum = joinedConsum.selectExpr("BuildingID", "Timestamp"
maximumsRatio = joinedConsum.group